# 1. Cleaning a Molecular Dataset

Before training or fine-tuning any model, your molecular dataset needs to be **cleaned**.
Raw SMILES strings from databases often contain entries that are:
- Chemically invalid (cannot be parsed).
- Too large or too small to be drug-like.
- Unsupported by the model's tokeniser (contain rare or unusual chemical groups).
- Containing stereochemistry, which is removed for simplicity.

This notebook walks through the cleaning pipeline and produces one output file:
- `cleaned.csv` — all molecules that passed both cleaning checks and the length filter

Splitting into train / val / test is handled separately in **Notebook 2**.


----
**Notebook structure:**
1. Imports & setup
2. Load your dataset
3. Clean the SMILES
4. Filter by length
5. Save the cleaned dataset
6. ✏️ Exercises

---
## Step 1 — Imports & Setup

We use the following Python packages:
- `pandas` for reading and manipulating the dataset
- `os` for building file paths
- `smiles_processing`, a provided module with two functions:
  - `clean_smiles(smiles)` standardises a SMILES string (removes salts, normalises charges, etc.). Returns `None` if the molecule cannot be cleaned.
  - `is_supported_chemical(smiles)` checks whether all tokens in the SMILES are part of the model vocabulary. Returns `True` or `False`.

In [ ]:
import os
import pandas as pd

# Provided module
# clean_smiles: standardises a SMILES string, returns None if it fails
# is_supported_chemical: checks whether the SMILES uses only supported tokens, i.e. those in the model's vocabulary
from scripts.smiles_processing import clean_smiles, is_supported_chemical

print("All imports successful!")

In [ ]:
# Set the base directory to the current working directory (where this notebook is located)
base_dir = os.path.dirname(os.path.abspath("__file__"))

---
## Step 2 — Load Your Dataset

Your dataset should be stored in a CSV file, containing at least one column with SMILES strings.

Your job: update the two variables in the following cell:
- `csv_path` — path to your input CSV file
- `smiles_column` — the name of the column that contains the SMILES strings

In [ ]:
# Edit these variables to match your dataset and preferences:
csv_path = os.path.join(base_dir, "your_path_here")                 # Please insert the path to your CSV file here
smiles_column = "SMILES"                                            # Name of the column containing SMILES strings
output_dir = os.path.join(base_dir, "your_path_here")               # Folder where outputs will be saved

output_dir = os.path.join(base_dir, "dataset", "cleaned_dataset")
os.makedirs(output_dir, exist_ok=True)

# Read the CSV into a DataFrame
df = pd.read_csv(
    csv_path
)
df.dropna(subset=[smiles_column], inplace=True)                     # Drop rows where the SMILES column is NaN

print(f"Loaded {len(df)} rows from {csv_path}.")
print(f"Columns: {list(df.columns)}")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Dataset shuffled.")
df.head()

---
## Step 3 — Clean the SMILES

We apply two checks to every SMILES string:

1. **`clean_smiles()`** — standardises the SMILES (e.g. removes counterions, normalises tautomers). If a molecule cannot be standardised at all, it returns `None`.
2. **`is_supported_chemical()`** — checks that every token in the cleaned SMILES belongs to the model's vocabulary. Molecules with unsupported tokens (e.g. unusual metals or rare bond types) cannot be encoded and must be removed.

Only molecules that pass **both** checks are kept.

In [ ]:
def is_supported_smiles(smiles):
    """
    Applies cleaning and support check to a single SMILES string.

    Parameters
    ----------
    smiles : str
        Raw SMILES string from the dataset.

    Returns
    -------
    cleaned : str or None
        The cleaned SMILES string, or None if cleaning failed.
    is_supported : int
        1 if the molecule is valid and supported, 0 otherwise.
        (We use an integer rather than a boolean so we can easily filter with pandas later.)
    """
    # Step 1: clean the SMILES — standardise and remove salts/counterions
    # to_canonical=False keeps the non-canonical form; the model handles canonicalisation itself
    cleaned = clean_smiles(smiles, to_canonical=False)

    # Step 2: check support — only mark as supported if cleaning succeeded AND
    # every token in the SMILES is in the model vocabulary
    supported = int(cleaned is not None and is_supported_chemical(cleaned))

    return cleaned, supported

In [ ]:
# Apply the cleaning function to every row in the dataset
# zip(*...) unpacks the two return values into two separate columns
df["cleaned_smiles"], df["is_supported"] = zip(
    *df[smiles_column].apply(is_supported_smiles)
)

In [ ]:
# Do some analysis on your dataset and count how many molecules passed and failed.
n_supported = df["is_supported"].sum()
n_unsupported = len(df) - n_supported

print(f"Total molecules:        {len(df)}")
print(f"Supported (kept):       {n_supported}  ({n_supported / len(df) * 100:.1f}%)")
print(f"Unsupported (removed):  {n_unsupported}  ({n_unsupported / len(df) * 100:.1f}%)")

In [ ]:
# Keep only the molecules that passed both checks
# We also select only the cleaned_smiles column — we no longer need the original or the flag
clean_df = df.query("is_supported == 1")[["cleaned_smiles"]]

print(f"Rows remaining after cleaning: {len(clean_df)}")
clean_df.head()

---
## Step 4 — Filter by Length

SMILES length is a proxy for molecular size. We remove molecules that are:
- **Too long** (> 150 characters) — very large molecules are harder to model and less likely to be drug-like
- **Too short** (≤ 6 characters) — very small fragments are not useful for drug discovery

You can adjust these thresholds in the exercise section if you want to explore the effect.

In [ ]:
# ── EDIT THESE if needed ───────────────────────────────────────────────────
min_length = 6  # Molecules with SMILES length <= this value are removed
max_length = 150  # Molecules with SMILES length > this value are removed
# ───────────────────────────────────────────────────────────────────────────

before = len(clean_df)

# Keep only molecules whose SMILES length is within the accepted range
# .map(len) computes the character length of each SMILES string
clean_df = clean_df[
    clean_df["cleaned_smiles"].map(len) <= max_length
]  # Remove too long
clean_df = clean_df[
    clean_df["cleaned_smiles"].map(len) > min_length
]  # Remove too short

after = len(clean_df)

print(f"Rows before length filter: {before}")
print(f"Rows after length filter:  {after}  ({before - after} removed)")

# Quick look at the length distribution of what remains
lengths = clean_df["cleaned_smiles"].map(len)
print(
    f"SMILES length — min: {lengths.min()}, max: {lengths.max()}, mean: {lengths.mean():.1f}"
)

#### Is your dataset large enough after cleaning?

You need at least **~50 molecules** for fine-tuning to be meaningful.
If your dataset is too small after cleaning, ask one of the instructors — we can help.

---
## Step 5 — Save the Cleaned Dataset

We rename the cleaned SMILES column to `SMILES` for consistency across all notebooks,
then save the result as a single CSV file.  Splitting into train / val / test happens
in Notebook 2.

`index=False` prevents pandas from writing row numbers as an extra column.

In [ ]:
# Rename column to 'SMILES' for consistency with the rest of the codebase
clean_df = clean_df.rename(columns={"cleaned_smiles": "SMILES"})

output_path = os.path.join(output_dir, "cleaned.csv")
clean_df.to_csv(output_path, index=False)

print(f"Saved {len(clean_df)} cleaned molecules to: {output_path}")

---
## ✏️ Exercise — Modify the Cleaning Pipeline

### Exercise 1 — Change the length thresholds
Go back to **Step 4** and try:
- `max_length = 100` — how many more molecules are removed?
- `min_length = 10` — does this remove many molecules from your dataset?

Re-run Steps 4 and 5. Does tightening the filter change the character of your dataset?

In [ ]:
# Your workspace — try the exercises here!
